[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_mnist_rete_neurale_didattica_momentum.ipynb)

# Una rete neurale impara MNIST

**Punto di partenza.** Nel notebook *MNIST con similarità coseno* abbiamo riconosciuto cifre scritte a mano usando 10 immagini-prototipo e il prodotto scalare normalizzato. Accuratezza sul test set: **~82%**. Un risultato dignitoso ma chiaramente limitato — il modello era pochissimo più sofisticato di un confronto pixel-per-pixel.

**Cosa abbiamo imparato dalle porte logiche.** Nei tre notebook sulle porte (OR lineare, NOR affine, XOR con attivazione) abbiamo visto che senza **non-linearità** una rete profonda è matematicamente equivalente a un singolo neurone affine. Per imparare problemi più complicati serve introdurre tra uno strato e l'altro una funzione di attivazione non lineare — la più semplice è la **ReLU**.

**Cosa facciamo oggi.** Costruiamo una **vera rete neurale** per MNIST: tre strati (784 → 128 → 64 → 10), ReLU come attivazione, training con gradient descent + momentum, loss cross-entropy. Vedremo che l'accuratezza fa un salto enorme rispetto al baseline cosine (dall'82% al ~97%). Alla fine c'è di nuovo il pannello-disegno, ma questa volta il riconoscitore sarà la **rete addestrata**.

## Setup

In [ ]:
!pip install --upgrade --no-cache-dir oli_ai > /dev/null

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad, jit
from jax.nn import relu, one_hot, softmax, log_softmax
import numpy as np
import matplotlib.pyplot as plt
from oli_ai.mnist_lib import carica_mnist, show_nn_graph, mostra_cifra

## 1. I dati — MNIST

Stessi dati del notebook precedente: **60.000** immagini di training e **10.000** di test, ognuna una matrice 28×28 di pixel da 0 (bianco) a 255 (nero).

Per dare i dati in pasto a una rete neurale facciamo due trasformazioni:

1. **Appiattiamo** ogni immagine in un vettore di 784 numeri (concateniamo le righe — è la stessa idea già vista nel notebook cosine).
2. **Normalizziamo** i pixel da 0..255 a 0..1 dividendo per 255. È un'abitudine standard: lavorare con numeri piccoli e di scala simile rende il training più stabile.

In [ ]:
immagini_train, etichette_train, immagini_test, etichette_test = carica_mnist()

X_train = jnp.asarray(immagini_train.reshape(-1, 784).astype('float32') / 255.0)
X_test  = jnp.asarray(immagini_test.reshape(-1, 784).astype('float32') / 255.0)
y_train = jnp.asarray(etichette_train)
y_test  = jnp.asarray(etichette_test)

print(f'X_train shape: {X_train.shape}    y_train shape: {y_train.shape}')
print(f'X_test  shape: {X_test.shape}     y_test  shape: {y_test.shape}')

# Vediamo una cifra di esempio
mostra_cifra(immagini_train[0])
print(f'etichetta: {int(y_train[0])}')

## 2. L'architettura della rete

Costruiamo una rete con tre strati:

- **input**: 784 neuroni (uno per pixel)
- **primo strato nascosto**: 128 neuroni
- **secondo strato nascosto**: 64 neuroni
- **output**: 10 neuroni (uno per cifra)

Tra uno strato e l'altro applichiamo la funzione di attivazione **ReLU** (la rettificazione `max(0, x)` che abbiamo conosciuto nel notebook XOR).

In [ ]:
show_nn_graph([784, 128, 64, 10], max_neurons=8)

# Conteggio dei parametri della rete
# - 784 -> 128: W1 (784,128) = 100352 pesi, b1 (128,) = 128 bias
# - 128 -> 64:  W2 (128, 64) =   8192 pesi, b2  (64,) =  64 bias
# - 64  -> 10:  W3 (64,  10) =    640 pesi, b3  (10,) =  10 bias
pesi  = 784*128 + 128*64 + 64*10
bias  = 128 + 64 + 10
print(f'Neuroni:    {784 + 128 + 64 + 10}')
print(f'Pesi:       {pesi:,}')
print(f'Bias:       {bias:,}')
print(f'Parametri:  {pesi + bias:,}')

## 3. Il modello (con attivazione ReLU)

La funzione `predict` calcola il forward pass della rete in tre passi:

1. Primo strato:  `h1 = ReLU(X · W1 + b1)`  →  (N, 128)
2. Secondo strato: `h2 = ReLU(h1 · W2 + b2)` →  (N, 64)
3. Output:         `out = h2 · W3 + b3`     →  (N, 10)

L'output ha 10 numeri per immagine: ognuno è un "punteggio" per una cifra. La cifra predetta sarà quella con il punteggio più alto (`argmax`).

In [ ]:
attivazione = relu

def predict(params, X):
    W1, b1, W2, b2, W3, b3 = params
    h1 = attivazione(X @ W1 + b1)
    h2 = attivazione(h1 @ W2 + b2)
    return h2 @ W3 + b3

def predici_classi(params, X):
    # prende i 10 punteggi e restituisce l'indice del massimo
    return jnp.argmax(predict(params, X), axis=1)

def accuratezza(params, X, y):
    # frazione di predizioni corrette (fra 0 e 1)
    return float(jnp.mean(predici_classi(params, X) == y))

## 4. La loss — softmax + cross-entropy

Per il training abbiamo bisogno di una "risposta giusta" da confrontare con l'output della rete (10 numeri per immagine). Codifichiamo le etichette in **one-hot**: l'etichetta `3` diventa il vettore `[0,0,0,1,0,0,0,0,0,0]`, l'etichetta `7` diventa `[0,0,0,0,0,0,0,1,0,0]`, e così via.

Nei notebook precedenti (porte logiche) abbiamo usato la **MSE** come loss. Per la classificazione, però, c'è una loss che funziona molto meglio: la **cross-entropy**. Si calcola in due passi:

**1. softmax.** La rete produce 10 punteggi liberi (numeri qualsiasi, anche negativi). La funzione **softmax** li trasforma in 10 **probabilità** che sommano a 1 — la rete dice *"sono al 73% sicura che sia un 3, al 18% un 5, ..."*. La formula: ogni punteggio è esponenziato e diviso per la somma di tutti gli esponenziali.

**2. cross-entropy.** È il **negativo del logaritmo** della probabilità assegnata alla cifra **giusta**:

- Se la rete dice 99% per la cifra giusta, loss = -log(0.99) ≈ 0.01 (premio!)
- Se la rete dice 50% per la cifra giusta, loss = -log(0.5) ≈ 0.69
- Se la rete dice 1% per la cifra giusta, loss = -log(0.01) ≈ 4.6 (punizione forte!)

La cross-entropy **premia molto le predizioni confidenti corrette** e **punisce molto le confidenti sbagliate**, e — fatto importante — i suoi gradienti restano forti anche quando il modello è vicino alla risposta giusta. Con la MSE invece il gradiente si spegne progressivamente avvicinandosi al target, e il modello smette di migliorare. Con la cross-entropy passeremo dal ~93% (cap della MSE su questa architettura) al ~97%.

In [ ]:
y_train_oh = one_hot(y_train, 10)
print(f'y_train_oh shape: {y_train_oh.shape}')
print(f'esempio: la cifra {int(y_train[0])} diventa {y_train_oh[0]}')

def loss(params, X, y_oh):
    # cross-entropy: -media( somma_k  y_oh[i,k] * log(softmax(predict)[i,k]) )
    # usiamo log_softmax direttamente perche' e' numericamente piu' stabile di log(softmax(...))
    return -jnp.mean(jnp.sum(y_oh * log_softmax(predict(params, X)), axis=1))

## 5. Inizializzazione dei parametri

Come nel notebook XOR, **non possiamo partire da zero** — i neuroni di ogni strato sono tutti uguali e i gradienti restano bloccati. Inizializziamo con piccoli numeri casuali.

Per le reti con ReLU c'è una formula standard che funziona molto bene, chiamata **inizializzazione di He**: i pesi si estraggono da una distribuzione normale con deviazione standard $\sqrt{2/n_\text{input}}$, dove $n_\text{input}$ è il numero di input di quello strato. È un dettaglio tecnico, ma rende il training molto più rapido e stabile.

In [ ]:
key = jr.PRNGKey(42)
k1, k2, k3 = jr.split(key, 3)

W1 = jr.normal(k1, (784, 128)) * jnp.sqrt(2/784)
b1 = jnp.zeros(128)
W2 = jr.normal(k2, (128, 64))  * jnp.sqrt(2/128)
b2 = jnp.zeros(64)
W3 = jr.normal(k3, (64, 10))   * jnp.sqrt(2/64)
b3 = jnp.zeros(10)

params = [W1, b1, W2, b2, W3, b3]
print(f'parametri totali: {sum(p.size for p in params):,}')
print(f'L iniziale: {float(loss(params, X_train, y_train_oh)):.4f}')

## 6. Il gradiente e il loop di addestramento (con **momentum**)

Il `grad` di JAX calcola le derivate parziali della loss rispetto a ognuno dei 109.386 parametri della rete. Una volta che abbiamo i gradienti facciamo un piccolo upgrade rispetto ai notebook precedenti.

Invece di aggiornare i parametri direttamente con `params = params - lr * g`, usiamo un'idea che si chiama **momentum**: accumuliamo una specie di **memoria** dei gradienti precedenti — pensala come una **pallina che rotola in discesa** e mantiene un po' della sua velocità anche quando incontra una salita lieve. La velocità non viene "resettata" a ogni passo.

In formula, manteniamo una variabile `v` (velocità) per ogni parametro:

```python
v = beta * v + g              # nuova velocita' = inerzia + gradiente nuovo
params = params - lr * v      # aggiorniamo coi v invece che con g diretto
```

Con `beta = 0.9` la pallina conserva il 90% della velocità precedente passo dopo passo. Risultato: il training accelera molto (le valli larghe si attraversano senza zig-zag, le discese ripide diventano più veloci) e si arriva a un'accuratezza molto più alta nello stesso tempo.

Usiamo `jit` per compilare la funzione di gradiente — con 109 mila parametri e 60 mila immagini, senza `jit` ogni epoca sarebbe lentissima.

In [ ]:
grad_loss = jit(grad(loss))
loss_jit  = jit(loss)              # anche la loss compilata, per velocita'

lr       = 0.1
beta     = 0.9                  # quanto si "ricorda" della velocita' precedente
n_epoche = 300
storia_loss = []

# velocita' iniziale = 0 per ogni parametro (la pallina parte ferma)
v = [jnp.zeros_like(p) for p in params]

for epoca in range(n_epoche):
    perdita = loss_jit(params, X_train, y_train_oh)
    g       = grad_loss(params, X_train, y_train_oh)
    storia_loss.append(float(perdita))

    if epoca % 30 == 0 or epoca == n_epoche - 1:
        acc = accuratezza(params, X_test, y_test)
        print(f'epoca {epoca:>3}: loss = {float(perdita):.4f}   accuratezza test = {acc*100:.2f}%')

    # update con momentum
    for i in range(len(params)):
        v[i]      = beta * v[i] + g[i]        # velocita' aggiornata
        params[i] = params[i] - lr * v[i]      # passo nella direzione della velocita'

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(storia_loss, color='#38bdf8')
plt.xlabel('epoca'); plt.ylabel('loss'); plt.title('Andamento della loss durante il training')
plt.grid(alpha=0.3); plt.show()

## 7. Le predizioni e l'accuratezza

Per fare una predizione su un'immagine: passiamo `X` alla rete, otteniamo 10 punteggi (uno per cifra), e prendiamo l'`argmax`. Misuriamo l'accuratezza sul training e sul test set.

In [ ]:
acc_train = accuratezza(params, X_train, y_train)
acc_test  = accuratezza(params, X_test,  y_test)
print(f'Accuratezza training set: {acc_train*100:.2f}%')
print(f'Accuratezza test set:     {acc_test*100:.2f}%')

## 8. Confronto con la cosine similarity

| Modello | Parametri | Accuratezza test |
|---|---|---|
| Cosine similarity (10 medie) | 7.840 | ~82% |
| Rete neurale 784-128-64-10 + ReLU + cross-entropy + momentum | **109.386** | ~97% |

La rete neurale ha **~14 volte più parametri** del baseline cosine. In cambio guadagna molti punti di accuratezza. La differenza fondamentale: il baseline cosine confronta i pixel grezzi con un prototipo, mentre la rete **impara feature intermedie** (negli strati nascosti da 128 e 64 neuroni) che descrivono le cifre in modo più astratto del pixel-per-pixel.

## 9. Prova tu — disegna una cifra

Esegui la cella qui sotto (il codice è nascosto: clicca la freccetta se vuoi guardarlo, ma non è necessario). Comparirà un pannello con un riquadro bianco: disegna dentro una cifra (0–9) e clicca **Inferisci**. La **rete neurale che hai appena addestrato** ti dirà cosa pensa di vedere, e ti mostrerà la sua "fiducia" per ognuna delle 10 cifre.

Come nel notebook cosine, prova a disegnare **grande e centrato** per ottenere predizioni più affidabili.

In [ ]:
#@title Pannello di disegno — clicca per avviare {display-mode: "form"}

# Cella tecnica: canvas HTML/JavaScript + decoder Python + chiamata alla rete.
# Non e' necessario leggerla.

import base64
import io
import numpy as np
import jax.numpy as jnp
from jax.nn import softmax
from PIL import Image
from IPython.display import HTML, display, JSON
from google.colab import output
from scipy.ndimage import center_of_mass, shift as _shift


_HTML_PANNELLO = """
<div style="font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 920px;">
  <p style="font-size: 16px;">
    Disegna una cifra (0-9) grande e centrata, poi premi <b>Inferisci</b>:
  </p>

  <div style="display: flex; gap: 24px; align-items: flex-start; flex-wrap: wrap;">

    <div>
      <canvas
        id="mnist-canvas"
        width="280"
        height="280"
        style="
          border: 2px solid #444;
          border-radius: 8px;
          background: white;
          touch-action: none;
          cursor: crosshair;
        ">
      </canvas>

      <div style="margin-top: 12px;">
        <button id="inferisci-btn" style="font-size: 16px; padding: 8px 14px;">Inferisci</button>
        <button id="cancella-btn" style="font-size: 16px; padding: 8px 14px; margin-left: 8px;">Cancella</button>
      </div>

      <div id="risultato" style="margin-top: 18px; font-size: 20px; font-weight: 700;">
        -
      </div>

      <div style="margin-top: 18px;">
        <div style="font-weight: 700; margin-bottom: 6px;">Quello che la rete vede (28×28):</div>
        <img
          id="preview-28"
          width="140"
          height="140"
          style="
            border: 1px solid #aaa;
            image-rendering: pixelated;
            background: black;
          ">
      </div>

      <div id="debug" style="margin-top: 12px; font-family: monospace; font-size: 13px; color: #555;"></div>
    </div>

    <div style="flex: 1; min-width: 320px;">
      <div style="font-weight: 700; margin-bottom: 8px;">Fiducia per ogni cifra:</div>
      <div id="barre">
        <div style="color: #888; padding: 12px 0;">
          Premi <b>Inferisci</b> per vedere quanto la rete e\u0301 sicura di ogni cifra.
        </div>
      </div>
    </div>

  </div>

  <p style="margin-top: 18px; color: #555;">
    Suggerimento: occupa la maggior parte del riquadro, tratto spesso, posizione centrale.
  </p>
</div>

<script>
(function() {
  const canvas = document.getElementById("mnist-canvas");
  const ctx = canvas.getContext("2d");

  const risultato = document.getElementById("risultato");
  const barre = document.getElementById("barre");
  const preview = document.getElementById("preview-28");
  const debug = document.getElementById("debug");

  const PLACEHOLDER_BARRE =
    '<div style="color: #888; padding: 12px 0;">' +
    'Premi <b>Inferisci</b> per vedere quanto la rete e\u0301 sicura di ogni cifra.' +
    '</div>';

  let drawing = false;
  let lastX = 0;
  let lastY = 0;

  function resetCanvas() {
    ctx.save();
    ctx.globalCompositeOperation = "source-over";
    ctx.fillStyle = "white";
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.restore();

    ctx.strokeStyle = "black";
    ctx.lineWidth = 20;
    ctx.lineCap = "round";
    ctx.lineJoin = "round";

    risultato.textContent = "-";
    barre.innerHTML = PLACEHOLDER_BARRE;
    preview.removeAttribute("src");
    debug.textContent = "";
  }

  function getPos(event) {
    const rect = canvas.getBoundingClientRect();
    if (event.touches && event.touches.length > 0) {
      return {
        x: event.touches[0].clientX - rect.left,
        y: event.touches[0].clientY - rect.top
      };
    }
    return {
      x: event.clientX - rect.left,
      y: event.clientY - rect.top
    };
  }

  function startDraw(event) {
    event.preventDefault();
    drawing = true;
    const pos = getPos(event);
    lastX = pos.x;
    lastY = pos.y;
    ctx.beginPath();
    ctx.arc(lastX, lastY, ctx.lineWidth / 2, 0, 2 * Math.PI);
    ctx.fillStyle = "black";
    ctx.fill();
  }

  function draw(event) {
    if (!drawing) return;
    event.preventDefault();
    const pos = getPos(event);
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(pos.x, pos.y);
    ctx.stroke();
    lastX = pos.x;
    lastY = pos.y;
  }

  function stopDraw(event) {
    event.preventDefault();
    drawing = false;
  }

  canvas.addEventListener("mousedown", startDraw);
  canvas.addEventListener("mousemove", draw);
  canvas.addEventListener("mouseup", stopDraw);
  canvas.addEventListener("mouseleave", stopDraw);
  canvas.addEventListener("touchstart", startDraw, {passive: false});
  canvas.addEventListener("touchmove", draw, {passive: false});
  canvas.addEventListener("touchend", stopDraw, {passive: false});
  canvas.addEventListener("touchcancel", stopDraw, {passive: false});

  document.getElementById("cancella-btn").onclick = resetCanvas;

  document.getElementById("inferisci-btn").onclick = async function() {
    risultato.textContent = "Inferenza in corso...";
    barre.innerHTML = "";
    debug.textContent = "";

    const dataURL = canvas.toDataURL("image/png");
    const res = await google.colab.kernel.invokeFunction(
      "notebook.disegno_predici",
      [dataURL],
      {}
    );

    const payload = res.data["application/json"];

    if (payload.cifra < 0) {
      risultato.textContent = "Disegna prima una cifra.";
      barre.innerHTML = PLACEHOLDER_BARRE;
      return;
    }

    risultato.textContent =
      "Predizione: " + payload.cifra +
      " — fiducia: " + (100 * Math.max(...payload.fiducia)).toFixed(1) + "%";

    if (payload.preview) {
      preview.src = payload.preview;
    }

    if (payload.debug) {
      debug.textContent = payload.debug;
    }

    barre.innerHTML = "";

    for (let i = 0; i < 10; i++) {
      const p = payload.fiducia[i];
      const row = document.createElement("div");
      row.style.display = "flex";
      row.style.alignItems = "center";
      row.style.margin = "4px 0";
      row.style.gap = "8px";

      const label = document.createElement("div");
      label.textContent = i;
      label.style.width = "20px";
      label.style.fontWeight = "700";

      const outer = document.createElement("div");
      outer.style.flex = "1";
      outer.style.height = "18px";
      outer.style.background = "#eee";
      outer.style.borderRadius = "999px";
      outer.style.overflow = "hidden";

      const inner = document.createElement("div");
      inner.style.width = (100 * p).toFixed(1) + "%";
      inner.style.height = "100%";
      inner.style.background = i === payload.cifra ? "#2563eb" : "#94a3b8";

      const pct = document.createElement("div");
      pct.textContent = (100 * p).toFixed(1) + "%";
      pct.style.width = "58px";
      pct.style.fontFamily = "monospace";

      outer.appendChild(inner);
      row.appendChild(label);
      row.appendChild(outer);
      row.appendChild(pct);
      barre.appendChild(row);
    }
  };

  resetCanvas();
})();
</script>
"""


def pannello_disegno_rete(params, predict_fn):
    """Pannello interattivo: disegna col mouse, premi 'Inferisci', vedi
    cosa predice la rete neurale per ognuna delle 10 cifre."""

    def _predici_da_canvas(data_url):
        # 1. Data URL -> immagine RGBA.
        _, b64 = data_url.split(',', 1)
        img_rgba = Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGBA')

        # 2. Componiamo eventuale trasparenza sopra uno sfondo bianco reale.
        sfondo = Image.new('RGBA', img_rgba.size, (255, 255, 255, 255))
        sfondo.alpha_composite(img_rgba)

        # 3. Grayscale.
        img = sfondo.convert('L')

        # 4. Il pannello disegna nero su bianco; MNIST e' sfondo nero=0,
        #    cifra chiara=255. Invertiamo.
        arr = (255 - np.array(img)).astype('uint8')

        # 5. Bounding box della cifra (sentinella -1 se vuoto).
        mask = arr > 30
        if not mask.any():
            return JSON({'cifra': -1, 'fiducia': []})

        rs = np.where(mask.any(axis=1))[0]
        cs = np.where(mask.any(axis=0))[0]
        cropped = arr[rs[0]:rs[-1] + 1, cs[0]:cs[-1] + 1]

        # 6. Resize in un box 20x20 preservando proporzioni (preprocessing MNIST).
        h, w = cropped.shape
        if h >= w:
            new_h = 20
            new_w = max(1, int(round(w * 20 / h)))
        else:
            new_w = 20
            new_h = max(1, int(round(h * 20 / w)))
        resized = np.array(
            Image.fromarray(cropped).resize((new_w, new_h), Image.BICUBIC),
            dtype='float64'
        )

        # 7. Inseriamo in un canvas 28x28 nero.
        canvas28 = np.zeros((28, 28), dtype='float64')
        y0 = (28 - new_h) // 2
        x0 = (28 - new_w) // 2
        canvas28[y0:y0 + new_h, x0:x0 + new_w] = resized

        # 8. Centriamo per centro di massa.
        cy, cx = center_of_mass(canvas28)
        if not np.isnan(cx) and not np.isnan(cy):
            canvas28 = _shift(
                canvas28,
                [13.5 - cy, 13.5 - cx],
                order=1,
                mode='constant',
                cval=0
            )
        canvas28 = np.clip(canvas28, 0, 255)

        # 9. Anteprima del 28x28 che la rete vede.
        preview = Image.fromarray(canvas28.astype('uint8'))
        preview = preview.resize((140, 140), Image.NEAREST)
        buf = io.BytesIO()
        preview.save(buf, format='PNG')
        preview_dataurl = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

        # 10. Debug diagnostico.
        pixel_attivi = float((canvas28 > 30).mean())
        debug = (
            f"canvas28 min={canvas28.min():.1f}, "
            f"max={canvas28.max():.1f}, "
            f"mean={canvas28.mean():.1f}, "
            f"pixel>30={pixel_attivi:.3f}, "
            f"bbox={cropped.shape}, "
            f"resize=({new_h},{new_w})"
        )

        # 11. Normalizza 0-1, appiattisci, passa alla rete.
        x_vec = (canvas28.flatten() / 255.0).astype('float32')
        x_batch = jnp.asarray(x_vec[None, :])  # (1, 784)
        scores = predict_fn(params, x_batch)[0]  # logits, shape (10,)
        fiducia = softmax(scores)
        cifra = int(jnp.argmax(scores))

        return JSON({
            'cifra': cifra,
            'fiducia': [float(f) for f in fiducia],
            'preview': preview_dataurl,
            'debug': debug
        })

    output.register_callback('notebook.disegno_predici', _predici_da_canvas)
    display(HTML(_HTML_PANNELLO))


pannello_disegno_rete(params, predict)


## Riepilogo del percorso

Questo è l'ultimo notebook del corso. Sintesi:

| Tappa | Tecnica | Accuratezza MNIST |
|---|---|---|
| Vettori + similarità coseno (10 medie) | matematica di base, no training | ~82% |
| Porte logiche OR / NOR | un neurone con gradient descent | — (didattica) |
| Porta XOR | rete `[2,8,1]` con ReLU | — (didattica) |
| **MNIST con rete neurale (questo)** | rete `[784,128,64,10]` con ReLU, cross-entropy e momentum, 109.386 parametri | ~97% |

L'idea è la stessa in tutto il corso: parametri appresi minimizzando una loss con gradient descent. Quello che cambia è **quanti parametri** e **quanto sono espressivi** (lineari → affini → non lineari → reti profonde). Più la rete è espressiva, più riesce a catturare strutture complesse — al costo di più parametri da imparare e training più sofisticati.

Tutto quello che vedete oggi nei modelli "famosi" (GPT, modelli di visione, sistemi di traduzione automatica) è la stessa identica idea: rete neurale + loss + gradient descent. La differenza è soltanto la **scala** — non più 109 mila parametri, ma centinaia di miliardi.